#Conjunto de dados de previsão de doenças cardíacas

##Visão geral do conjunto de dados

A Organização Mundial da Saúde estimou que 12 milhões de mortes ocorrem em todo o mundo, todos os anos, devido a doenças cardíacas. Metade das mortes nos Estados Unidos e em outros países desenvolvidos são devidas a doenças cardiovasculares. O prognóstico precoce das doenças cardiovasculares pode auxiliar na tomada de decisões sobre mudanças no estilo de vida em pacientes de alto risco e, por sua vez, reduzir as complicações.

**Fonte**:

 O conjunto de dados está disponível publicamente no site Kaggle e é proveniente de um estudo cardiovascular em andamento com residentes da cidade de Framingham, Massachusetts. O conjunto de dados fornece as informações dos pacientes durante acompanhamento de 10 anos e inclui mais de 4.000 registros e 15 atributos.

##Apresentação das variáveis

Cada atributo é um fator de risco potencial. Existem fatores de risco demográficos, comportamentais e médicos.

####**Demográfico**

- **male**: masculino =1 ou feminino = 0
- **age**: Idade do paciente

####**Comportamental**

- **currentSmoker**: se o paciente é fumante atual(=1) ou não (=0)
- **cigsPerDay**: o número de cigarros que a pessoa fumou em média em um dia.

####**Médico (contexto histórico)**

- **BPMeds**: se o paciente estava (=1) ou não (=0) tomando medicação para pressão arterial
- **prevalentStroke**: se o paciente já teve(=1) ou não (=0) um AVC
- **prevalentHyp**: se o paciente era hipertenso(=1) ou não(=0)
- **diabetes**: se o paciente tinha(=1) ou não diabetes (=0)

####**Médico (atual)**

- **totChol**: nível de colesterol total
- **sysBP**: pressão arterial sistólica
- **diaBP**: pressão arterial diastólica
- **BMI**: Índice de Massa Corporal
- **heartRate**: frequência cardíaca
- **glucose**: nível de glicose

####**Desenvolvimento Doença**
**TenYearCHD**: Doença cardíaca coronária em 10 anos (Sim = 1 , Não = 0)

##Objetivo da atividade:
Criar um modelo de aprendizado de máquina (problema de classificação) para identificar qual o percentual de chances de um paciente desenvolver doença cardíaca coronariana.

### Instalando os pacotes externos

In [2]:
%%capture
!pip install -U mlflow --quiet


Este pacote foi instalado para o rastreamento e gerenciamento deste experimento de Machine Learning, sendo permitido a realização do *autologging* dos parâmetros, métricas e do modelo treinado. Isto facilita a comparação entre diferentes execuções do modelo utilizado, compactuando com as boas práticas de MLOps.

### Importando as bibliotecas

In [25]:
import os
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

### Importando os dados

In [5]:
data = pd.read_csv('/content/framingham.csv')

In [6]:
data.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


Fazendo a contagem de quantos pacientes desenvolveram doença cardíaca (=1) e quantos não desenvolveram (=0)

In [7]:
data['TenYearCHD'].value_counts()

,count
TenYearCHD,
0,3594
1,644


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4238 non-null   int64  
 1   age              4238 non-null   int64  
 2   education        4133 non-null   float64
 3   currentSmoker    4238 non-null   int64  
 4   cigsPerDay       4209 non-null   float64
 5   BPMeds           4185 non-null   float64
 6   prevalentStroke  4238 non-null   int64  
 7   prevalentHyp     4238 non-null   int64  
 8   diabetes         4238 non-null   int64  
 9   totChol          4188 non-null   float64
 10  sysBP            4238 non-null   float64
 11  diaBP            4238 non-null   float64
 12  BMI              4219 non-null   float64
 13  heartRate        4237 non-null   float64
 14  glucose          3850 non-null   float64
 15  TenYearCHD       4238 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 529.9 KB


## Fazendo a transformação e separação dos dados

1) Identificando valores NaNs

In [9]:
data.isnull().sum()

,0
male,0
age,0
education,105
currentSmoker,0
cigsPerDay,29
BPMeds,53
prevalentStroke,0
prevalentHyp,0
diabetes,0
totChol,50


2) Removendo Valores NaNs

In [10]:
data_cleaned = data.dropna()

In [11]:
data_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3656 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             3656 non-null   int64  
 1   age              3656 non-null   int64  
 2   education        3656 non-null   float64
 3   currentSmoker    3656 non-null   int64  
 4   cigsPerDay       3656 non-null   float64
 5   BPMeds           3656 non-null   float64
 6   prevalentStroke  3656 non-null   int64  
 7   prevalentHyp     3656 non-null   int64  
 8   diabetes         3656 non-null   int64  
 9   totChol          3656 non-null   float64
 10  sysBP            3656 non-null   float64
 11  diaBP            3656 non-null   float64
 12  BMI              3656 non-null   float64
 13  heartRate        3656 non-null   float64
 14  glucose          3656 non-null   float64
 15  TenYearCHD       3656 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 485.6 KB


In [12]:
data_cleaned['TenYearCHD'].value_counts()

,count
TenYearCHD,
0,3099
1,557


Para poder dar andamento na preparação dos dados precisamos confirmar se os dados são do tipo DataFrame.

In [15]:
type(data_cleaned)

pandas.core.frame.DataFrame

Com esta confirmação, podemos separar o conjunto dos atributos (usados para treinar o modelo) e as variáveis alvo (o que o modelo tentará prever). Ou seja, todas as colunas contendo os dados demográficos, comportamentais e médicos dos pacientes estão selecionadas na variável X, enquanto a variável Y recebe apenas a última coluna.

In [16]:
X = data_cleaned.iloc[:, :-1]
y = data_cleaned.iloc[:, -1]

#### Transformando os dados

Esta transformação consiste na padronização dos dados por meio da criação da ferramenta *StandardScaler*. Essa ferramenta transforma os dados de forma que cada coluna/feature (característica considerada pelo paciente) tenha uma média 0 e um desvio padrão 1.

Assim, caso uma feature tenha um valor muito maior que outra, ela pode dominar a função e o algoritmo pode não aprender corretamente com as features com valores menores, diminuindo a influência. Desta maneira, esta ferramenta evita isto  e acaba melhorando o desempenho dos algoritmos de Machine Learning uma vez que evita distorções no modelo a ser desenvolvido.

In [17]:
scaler = StandardScaler()

Com a criação da ferramenta (scaler)  de padronização realizada, o fit(X) é etapa onde o scaler aprende os parâmetros do conjunto de dados (X). E em seguida, usa os parâmetros aprendidos para escalar os dados (transform(X)).

In [18]:
X = scaler.fit_transform(X)

#### Fazendo a separação em dados de treino e teste

Para realizar a separação foi selecionado 25% dos dados para o teste e 75% para o treino. Além disto, para facilitar a aleatoriedade da seleção dos dados, foi definida uma semente aleatória, garantindo o embaralhamento dos dados de maneira mais precisa e reprodutível.

In [21]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.25,
                                                    random_state=RANDOM_STATE)
print(f"Train data shape of X = {X_train.shape} and Y = {y_train.shape}")
print(f"Test data shape of X = {X_test.shape} and Y = {y_test.shape}")

Train data shape of X = (2742, 15) and Y = (2742,)
Test data shape of X = (914, 15) and Y = (914,)


### Criação e Treinamento do Modelo de Regressão Logística

 Por se tratar da criação de um modelo de classificação binária, onde o paciente pode desenvolver ou não a doença cardíaca, utilizou-se o algoritmo de Regressão Logística de classificação binária. Para isto, foi necessário primeiramente criar o modelo e depois treiná-lo de maneira que ele aprenda os padrões e relações entre as características e a resposta alvo.

Criando o modelo de Regressão Logística

In [22]:
log_reg = LogisticRegression()

Treinando o modelo de Regressão Logística

Para o treino do modelo foi necessário acrescentar os dados de entrada que o modelo precisa utilizar, que seria as 15 variáveis colhidas para cada um dos pacientes (X) e a identificação dos dados de saída (y), que neste caso seria a capacidade do paciente em desenvolver a doença cardíaca.

In [33]:
with mlflow.start_run():
    log_reg.fit(X = X_train ,
                y = y_train)


2026/04/15 20:24:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/15 20:24:21 INFO mlflow.store.db.utils: Updating database tables
2026/04/15 20:24:27 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/04/15 20:24:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Para identificar o quão bom este modelo treinado está, a métrica utilizada foi a **Acurácia**. Isto pode ser feito considerando ainda os dados de teste e seus respectivos resultados. Ou seja, eu utilizo uma amostra (x_test) que eu já sei o resultado e peço para ele prever os resultados (y_test).

In [24]:
y_pred = log_reg.predict(X = X_test)

accuracy_score(y_true = y_test,
               y_pred = y_pred)

0.8479212253829321

É possível verificar que o modelo consegue prever com 84,79% de acurária se um paciente pode desenvolver ou não doença coronariana. O que pode ser interpretado como uma boa métrica.

#### Rastreamento do experimento de Machine Learning

Para poder fazer o rastreamento do experimento, utilizou-se o registro automático (autologging) do experimento realizado pelo scikit-learn. Aqui, as informações salvas vão desde hiperparâmetros de modelo, como métricas de treinamento e validação, até o modelo em si.

Além disto, foi criado o diretório mlruns, que armazena os artefatos gerados pelo experimento, seja o model.pkl, o requirements.txt e o python_env. Outros arquivos são salvos tais como params, metrics.

Um ponto importante é que parâmetro se refere a tudo que o modelo aprende, enquanto Params é tudo aquilo que o Modelo utilizou. Para a produção, os arquivos model.pkl, requirements.txt e python_env são os arquivos necesários.

In [27]:
mlflow.sklearn.autolog(log_post_training_metrics=True)

###Pevisão para um exemplo

Abaixo é apresentado as características/features de um paciente aleatório:
- male: Masculino (= 1)
- Idade: 39 anos
- education: 4.0
- currentSmoker:  Fumante (=1)
- cigsPerDay: 5 cigarros por dia
- BPMeds: Não toma medicamento (=0)
- prevalentStroke: Já teve AVC (=1)
- prevalentHyp: Não é hipertenso (=0)
- diabetes: Não é diabético (=0)
- totChol: 200
- sysBP: 120
- diaBP: 80
- BMI: 24
- heartRate: 80
- glucose: 88

Traduzindo os dados do paciente para que o modelo criado consiga realizar a previsão.

Nota: Foi utilizado o reshape para garantir que o array seja bidimensional com uma única linha (1) e um número de coluna (-1).

In [35]:
me = np.array([1, 39, 4, 1, 5, 0, 1, 0, 0, 200, 120, 80, 24, 80, 88]).reshape(1, -1)

Para poder rodar array é necessario normalizar ela com a ferramenta scaler, criada anteriormente.

In [36]:
me_scaled = scaler.transform(me)

Fazendo a predição relacionando os dados deste paciente com o modelo criado

In [40]:
log_reg.predict(scaler.transform(me))

array([0])

Ou seja, o modelo consegue prever que o paciente utilizado no exemplo não irá desenvolver doença cardíaca coronariana em 10 anos.